## **Step 1: Dataset: Tiny Shakespeare**
There's a famous cleaned version of Shakespeare that's been the standard LLM toy dataset since Andrej Karpathy's char-RNN days. It's ~1MB, plain text, perfect for a small model.

In [ ]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)

with open("shakespeare.txt", "w") as f:
    f.write(response.text)

# Quick sanity check
text = response.text
print(f"Total characters: {len(text):,}")
print(f"First 300 chars:\n{text[:300]}")

Total characters: 1,115,394
First 300 chars:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## **Step 2: Tokenizer: Character-Level vs Subword**
A MiniModel was built with a simple vocabulary. Shakespeare gives two clean options:

- Character-level — vocab is just the ~65 unique characters in the text (a-z, A-Z, punctuation, space, newline). Simple, tiny vocab, works well with a small model.
- Subword (BPE) — you'd use tiktoken or sentencepiece. Richer tokens, but vocab size jumps to thousands. Your small model would struggle with a large vocab — the output head (d_model → vocab_size) becomes massive relative to everything else.

In [ ]:
# Build character-level vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
print(f"Vocab size: {vocab_size}")  # should be ~65

# Encoder: char → int
stoi = {ch: i for i, ch in enumerate(chars)}

# Decoder: int → char
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Test it
print(encode("Hello"))       # → [20, 43, 50, 50, 53]
print(decode([20, 43, 50, 50, 53]))  # → "Hello"

# Encode the full dataset
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}")  # → torch.Size([1115394])

Vocab size: 65
[20, 43, 50, 50, 53]
Hello
Data shape: torch.Size([1115394])


## **Step 3: Train/Val Split and Hyperparameters**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import requests

In [ ]:
# 90% train, 10% validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens  : {len(val_data):,}\n")

# Hyperparameters — tuned for a small model on Colab free tier
block_size = 64     # context window — T in (B, T)
batch_size = 32     # sequences per batch — B in (B, T)
d_model    = 128    # embedding dimension
n_heads    = 4      # attention heads  (d_model must be divisible by n_heads)
n_kv_heads = 2      # number of K/V head groups (must divide n_heads evenly)
# d_ff     = 512    # FFN inner dimension (4 × d_model is the standard)
d_ff       = 384    # SwiGLU uses 2 parallel projections, so scale to 2/3 × 512 ≈ 384
n_layers   = 4      # stacked transformer blocks
dropout    = 0.1    # 10% of activations zeroed during training
lr         = 3e-4   # Adam learning rate
max_iters  = 3000   # training steps
eval_every = 200    # print val loss every N steps

device     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}\n")

Train tokens: 1,003,854
Val tokens  : 111,540

Training on: cpu



- `block_size = 64` — the context window. Each training example is 64 characters long. The model sees 64 characters and predicts the next character at every position. This is T (seq_len) in your shape notation.
- `batch_size = 32` — 32 independent sequences processed simultaneously. This is B in your shapes. Higher = faster training but more GPU memory.
- `d_model = 128` — embedding dimension. Every token becomes a 128-dimensional vector. Smaller than GPT-2's 768 but fine for a character-level toy model.
- `n_heads = 4` — 4 attention heads. Each head gets d_model / n_heads = 128 / 4 = 32 dimensions. Important: d_model must be divisible by n_heads.
- `n_layers = 4` — 4 stacked transformer blocks. Each block refines the representations further.
- `dropout = 0.1` — randomly zeroes 10% of activations during training. Prevents overfitting by stopping the model from relying too heavily on any single pathway. Disabled during generation.
- `lr = 3e-4` — learning rate of 0.0003. This is the classic Adam learning rate for transformers — Karpathy calls it "the best learning rate." Not too fast (explodes), not too slow (crawls).
- `max_iters = 3000` — run 3000 training steps. Each step processes one batch of 32 sequences. Enough to see clear improvement on Colab's free T4 GPU in a few minutes.

## **Step 4: The Batch Loader**

In [ ]:
def get_batch(split):
    # Step 1: pick which river to fish from
    data_split = train_data if split == 'train' else val_data

    # Step 2: throw 32 random darts
    # Upper limit is len-block_size so we never fall off the end
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    # ix is just 32 random integers, e.g. [40231, 812445, 3891, ...]

    # Step 3: for each dart position, grab a 64-character window
    x = torch.stack([data_split[i   : i +  block_size   ] for i in ix])
    y = torch.stack([data_split[i+1 : i +  block_size+1 ] for i in ix])
    # x shape: (32, 64) — inputs
    # y shape: (32, 64) — targets (same windows, shifted right by 1)

    return x.to(device), y.to(device)

# Test it
xb, yb = get_batch('train')
print(f"Input shape:  {xb.shape}")   # → (32, 64)
print(f"Target shape: {yb.shape}")   # → (32, 64)

Input shape:  torch.Size([32, 64])
Target shape: torch.Size([32, 64])


## **Step 5: Positional Encoding**

In [ ]:
# Replaced by RoPE. Not called anywhere.
# def positional_encoding(seq_len, d_model, device):
#     """
#     Returns PE matrix of shape (seq_len, d_model).
#     Same sinusoidal formula as before — now takes device as arg
#     so it lands on the right hardware automatically.
#     """
#     pe = torch.zeros(seq_len, d_model, device=device)
#     for pos in range(seq_len):
#         for i in range(0, d_model, 2):
#             denom        = 10000 ** (i / d_model)
#             pe[pos, i]   = math.sin(pos / denom)
#             if i + 1 < d_model:
#                 pe[pos, i+1] = math.cos(pos / denom)
#     return pe

## **Step 5.5: RoPE — Rotary Positional Embeddings**
Replaces sinusoidal PE. Instead of adding position info to embeddings once
at the start, RoPE rotates Q and K vectors inside every attention layer,
right before the dot product. This means:
  - Position info is fresh at every layer (not diluted after 4 blocks)
  - The dot product automatically encodes *relative* distance, not absolute position
  - Works naturally at inference time with the KV cache

Two functions:
  - `precompute_rope_freqs` — runs once at model init, builds the fixed cos/sin tables
  - `apply_rope`            — called inside attention, rotates Q or K using those tables

In [ ]:
def precompute_rope_freqs(d_k, max_seq_len, base=10000, device='cpu'):
    """
    Precomputes the cosine and sine tables used for RoPE rotation.

    d_k        : dimension per head (e.g. 32 for d_model=128, n_heads=4)
    max_seq_len: maximum sequence length we'll ever see (use block_size here)
    base       : controls rotation speeds — 10000 is the standard default
                 (same value used in the original sinusoidal PE denominator)

    Returns:
        cos_table : (max_seq_len, d_k/2)  — one cos value per position per pair
        sin_table : (max_seq_len, d_k/2)  — one sin value per position per pair
    """

    # Step 1: compute one base angle (theta) per dimension PAIR
    # d_k=32 → 16 pairs → 16 theta values
    # Formula: theta_i = 1 / (base ^ (2i / d_k))  for i = 0, 1, ..., d_k/2 - 1
    # i=0  → theta = 1/10000^0       = 1.0      (fastest rotation)
    # i=1  → theta = 1/10000^(2/32)  = 0.66     (slightly slower)
    # ...
    # i=15 → theta = 1/10000^(30/32) = 0.000126 (slowest rotation)
    pair_indices = torch.arange(0, d_k, 2, device=device).float()  # [0, 2, 4, ..., d_k-2]
    thetas       = 1.0 / (base ** (pair_indices / d_k))             # (d_k/2,)

    # Step 2: for every token position, multiply by its theta
    # positions: [0, 1, 2, ..., max_seq_len-1]
    # result[pos, i] = pos * theta_i  ← rotation angle for token at 'pos', pair 'i'
    positions = torch.arange(max_seq_len, device=device).float()    # (max_seq_len,)
    angles    = torch.outer(positions, thetas)                       # (max_seq_len, d_k/2)
    # torch.outer: multiplies every position by every theta
    # angles[3, 1] = 3 * theta_1  → rotation angle for position 3, pair 1

    # Step 3: precompute cos and sin of every angle
    # (so we don't recompute them from scratch on every forward pass)
    cos_table = angles.cos()   # (max_seq_len, d_k/2)
    sin_table = angles.sin()   # (max_seq_len, d_k/2)

    return cos_table, sin_table

In [ ]:
def apply_rope(x, cos_table, sin_table, start_pos=0):
    """
    Rotates Q or K vectors using precomputed RoPE tables.

    x          : (B, num_heads, T, d_k) — Q or K after splitting into heads
    cos_table  : (max_seq_len, d_k/2)
    sin_table  : (max_seq_len, d_k/2)
    start_pos  : where in the sequence does this input start
                 (0 for training/prefill, grows during decode with KV cache)

    Returns:
        x_rotated : (B, num_heads, T, d_k) — same shape, rotated values
    """
    B, num_heads, T, d_k = x.shape

    # Step 1: slice out only the rows of cos/sin we need
    # (positions start_pos through start_pos + T - 1)
    cos = cos_table[start_pos : start_pos + T]   # (T, d_k/2)
    sin = sin_table[start_pos : start_pos + T]   # (T, d_k/2)

    # Step 2: split x into consecutive pairs along the last dimension
    # d_k=32 → x_even = dims [0,2,4,...,30], x_odd = dims [1,3,5,...,31]
    x_even = x[..., 0::2]   # (B, num_heads, T, d_k/2) — first of each pair
    x_odd  = x[..., 1::2]   # (B, num_heads, T, d_k/2) — second of each pair

    # Step 3: apply the 2D rotation formula to each pair
    # Standard 2D rotation of vector [a, b] by angle α:
    #   new_a = a*cos(α) - b*sin(α)
    #   new_b = a*sin(α) + b*cos(α)
    x_rotated_even = x_even * cos - x_odd * sin   # (B, num_heads, T, d_k/2)
    x_rotated_odd  = x_even * sin + x_odd * cos   # (B, num_heads, T, d_k/2)

    # Step 4: interleave even and odd back into one tensor
    # Stack along a new last dim → (B, num_heads, T, d_k/2, 2)
    # then flatten last two dims → (B, num_heads, T, d_k)
    x_rotated = torch.stack([x_rotated_even, x_rotated_odd], dim=-1)
    x_rotated = x_rotated.flatten(-2)             # (B, num_heads, T, d_k)

    return x_rotated

## **Step 6: Multi Head Attention — GQA (Grouped Query Attention)**

GQA sits between MHA (every head has its own K,V) and MQA (one K,V for all).
It defines n_kv_heads groups — each group shares one K,V pair among
(num_heads / n_kv_heads) query heads. Query heads stay fully independent.

Memory saving: KV cache stores n_kv_heads K,V matrices instead of num_heads.
For your model: 2 KV heads instead of 4 → 2× smaller KV cache.
For LLaMA 70B:  8 KV heads instead of 64 → 8× smaller KV cache.

Quality: near-MHA because each Q head still asks its own question via its
own W_Q. The shared K,V just needs to be rich enough to answer multiple
different questions — empirically, 1 KV per 4-8 Q heads is enough.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, n_kv_heads, dropout, max_seq_len):
        super().__init__()
        assert d_model % num_heads == 0,   "d_model must be divisible by num_heads"
        assert num_heads % n_kv_heads == 0, "num_heads must be divisible by n_kv_heads"

        self.d_model          = d_model
        self.num_heads        = num_heads
        self.n_kv_heads       = n_kv_heads
        self.heads_per_group  = num_heads // n_kv_heads  # Q heads sharing each KV pair
        self.d_k              = d_model // num_heads     # dim per head

        # W_Q projects to full num_heads dimension — every Q head is independent
        self.W_Q = nn.Linear(d_model, num_heads  * self.d_k, bias=False)
        # W_K and W_V project to n_kv_heads dimension — smaller than before
        # e.g. d_model=128, n_kv_heads=2, d_k=32 → projects to 64, not 128
        self.W_K = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_V = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

        # RoPE tables — precomputed once, fixed (not learned), moved to GPU with model
        cos_table, sin_table = precompute_rope_freqs(self.d_k, max_seq_len)
        self.register_buffer('cos_table', cos_table)  # (max_seq_len, d_k/2)
        self.register_buffer('sin_table', sin_table)  # (max_seq_len, d_k/2)

    def forward(self, x, kv_cache=None, start_pos=0):
        """
        x shape:
          Training / prefill : (B, T, d_model)
          Decode (1 new token): (B, 1, d_model)

        kv_cache: dict with 'K' and 'V' of shape (B, n_kv_heads, T_past, d_k)
                  or None on the first call.
        start_pos: position index where this input starts in the full sequence.

        Returns:
          output       : (B, T, d_model) — same shape as input
          new_kv_cache : updated cache dict with compact (n_kv_heads) K, V
        """
        B, T, C = x.shape

        # ── Project Q, K, V ─────────────────────────────────────────────────
        Q = self.W_Q(x)   # (B, T, num_heads * d_k)  = (B, T, 128)
        K = self.W_K(x)   # (B, T, n_kv_heads * d_k) = (B, T, 64)  ← smaller
        V = self.W_V(x)   # (B, T, n_kv_heads * d_k) = (B, T, 64)  ← smaller

        # ── Reshape into heads ───────────────────────────────────────────────
        # Q: split into num_heads (4) heads of size d_k (32)
        Q = Q.view(B, T, self.num_heads,  self.d_k).transpose(1, 2)  # (B, 4, T, 32)
        # K, V: split into n_kv_heads (2) groups of size d_k (32)
        K = K.view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)  # (B, 2, T, 32)
        V = V.view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)  # (B, 2, T, 32)

        # ── Apply RoPE to Q and K ────────────────────────────────────────────
        # Q gets 4-head rotation, K gets 2-head rotation.
        # Both use the same start_pos — same sequence positions, just different
        # numbers of heads. apply_rope reads shape from the tensor automatically.
        Q = apply_rope(Q, self.cos_table, self.sin_table, start_pos)  # (B, 4, T, 32)
        K = apply_rope(K, self.cos_table, self.sin_table, start_pos)  # (B, 2, T, 32)

        # ── KV Cache — store COMPACT K, V (n_kv_heads, not num_heads) ────────
        # The cache stores only 2 heads worth of K, V, not 4.
        # This is the memory saving — expansion happens at attention time below.
        if kv_cache is not None:
            K = torch.cat([kv_cache['K'], K], dim=2)   # (B, 2, T_past+T, 32)
            V = torch.cat([kv_cache['V'], V], dim=2)   # (B, 2, T_past+T, 32)

        new_kv_cache = {'K': K, 'V': V}   # shape: (B, n_kv_heads=2, full_len, d_k)
        full_len = K.shape[2]

        # ── Expand K, V to match num_heads before dot product ────────────────
        # repeat_interleave(n, dim=1) repeats each "slice" along dim 1, n times.
        # (B, 2, full_len, 32) → (B, 4, full_len, 32)
        # K_group0 becomes [K_group0, K_group0], K_group1 becomes [K_group1, K_group1]
        # so Q_head0 and Q_head1 both see K_group0, Q_head2 and Q_head3 see K_group1.
        K = K.repeat_interleave(self.heads_per_group, dim=1)  # (B, 4, full_len, 32)
        V = V.repeat_interleave(self.heads_per_group, dim=1)  # (B, 4, full_len, 32)

        # ── Everything below is identical to MHA ─────────────────────────────
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)  # (B, 4, T, full_len)

        if T > 1:
            mask = torch.triu(
                torch.ones(T, full_len, device=x.device),
                diagonal=full_len - T + 1
            ).bool()
            scores = scores.masked_fill(mask, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        out = attn_weights @ V                            # (B, 4, T, 32)

        # ── Merge heads back ─────────────────────────────────────────────────
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, 128)
        return self.W_O(out), new_kv_cache

## **Step 7: Feed Forward Network (FFN)**

In [ ]:
# class FeedForward(nn.Module):
#     def __init__(self, d_model, d_ff, dropout):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(d_model, d_ff),
#             nn.ReLU(),
#             nn.Linear(d_ff, d_model),
#             nn.Dropout(dropout),
#         )

#     def forward(self, x):
#         return self.net(x)  # works on any shape (..., d_model)

Replaces the single ReLU path with two parallel learned projections (SwiGLU):
- `W1 (gate path)`  — passed through Swish activation (= x * sigmoid(x))
- `W2 (value path)` — no activation, carries the raw content
- `W3 (contract)`   — projects the gated result back to d_model

The elementwise multiplication (gate * value) lets the network learn
*which* features to amplify (W1) and *what content* to carry (W2)
independently, then combine them. W1 and W2 are forced to specialise
because their gradients are cross-coupled through the multiplication.

`bias=False` throughout — consistent with LLaMA/Mistral/Qwen. With RMSNorm
applied before the FFN, the input is already normalised, so a learned
bias offset would be partially undone and adds little benefit.

d_ff should be pre-scaled to 2/3 of the original value (set in hyperparams)
so total parameter count stays comparable to the old single-path design.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.W1      = nn.Linear(d_model, d_ff, bias=False)  # gate path
        self.W2      = nn.Linear(d_model, d_ff, bias=False)  # value path
        self.W3      = nn.Linear(d_ff, d_model, bias=False)  # contract
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Gate path: Swish activation (F.silu is PyTorch's name for Swish = x * sigmoid(x))
        gate  = F.silu(self.W1(x))   # (B, T, d_ff) — decides what to amplify
        # Value path: no activation — carries the raw projected content
        value = self.W2(x)            # (B, T, d_ff)
        # Gated multiplication: gate controls how much of value passes through
        x = gate * value              # (B, T, d_ff) — elementwise
        # Contract back to d_model
        x = self.W3(x)                # (B, T, d_model)
        return self.dropout(x)

## **Step 7.5: RMSNorm**

Replaces LayerNorm. Drops mean-subtraction (re-centering) and keeps only the re-scaling step, using Root Mean Square instead of standard deviation.

Faster (one pass over the vector instead of two) with no loss in training stability — this is what LLaMA, Mistral, Qwen, and Gemma all use.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(d_model))  # learned scale

    def forward(self, x):
        # x shape: (..., d_model) — works for (B, T, d_model) during
        # training/prefill AND (B, 1, d_model) during single-token decode.
        rms    = x.pow(2).mean(dim=-1, keepdim=True).sqrt()   # RMS over last dim
        x_norm = x / (rms + self.eps)                         # re-scale only
        return self.gamma * x_norm                            # learned scale

## **Step 8: The Transformer Block**

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout, max_seq_len):
        super().__init__()
        self.attention    = MultiHeadAttention(d_model, num_heads, n_kv_heads, dropout, max_seq_len)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1        = RMSNorm(d_model)
        self.norm2        = RMSNorm(d_model)

    def forward(self, x, kv_cache=None, start_pos=0):
        # Pre-LN: normalize BEFORE feeding into sublayer
        attn_out, new_kv_cache = self.attention(self.norm1(x), kv_cache=kv_cache, start_pos=start_pos)
        x = x + attn_out                  # residual connection

        ff_out = self.feed_forward(self.norm2(x))
        x = x + ff_out                    # residual connection

        return x, new_kv_cache

## **Step 9: The Full MiniGPT Model**

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout, max_seq_len):
        super().__init__()
        self.embedding   = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks      = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout, max_seq_len)
            for _ in range(num_layers)
        ])
        self.final_norm  = RMSNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids, kv_caches=None, start_pos=0):
        """
        token_ids: (B, T)  during training
                   (B, T)  during prefill  (T = prompt length)
                   (B, 1)  during decode   (one new token per step)

        kv_caches: list of num_layers cache dicts, or None
        start_pos: where in the full sequence does this input begin
                   (0 for training/prefill, grows during decode)

        Returns:
            logits     : (B, T, vocab_size)
            kv_caches  : updated list of per-block caches
        """
        B, T = token_ids.shape

        if kv_caches is None:
            kv_caches = [None] * len(self.blocks)

        # Step 1: Embed tokens → (B, T, d_model)
        x = self.embedding(token_ids)
        x = self.emb_dropout(x)

        # NOTE: No positional encoding addition here anymore.
        # RoPE handles position inside each attention layer instead.
        # pe               = positional_encoding(start_pos + T, self.embedding.embedding_dim, token_ids.device)
        # pe_slice         = pe[start_pos : start_pos + T]    # (T, d_model)
        # x                = x + pe_slice                     # broadcasts over batch dim

        # Step 2: Pass through transformer blocks
        new_kv_caches = []
        for block, block_cache in zip(self.blocks, kv_caches):
            x, new_block_cache = block(x, kv_cache=block_cache)
            new_kv_caches.append(new_block_cache)

        # Step 3: Final norm + output head
        x      = self.final_norm(x)
        logits = self.output_head(x)      # (B, T, vocab_size)

        return logits, new_kv_caches

## **Step 10: Instantiate Model**

In [ ]:
model = MiniGPT(
    vocab_size  = vocab_size,
    d_model     = d_model,
    num_heads   = n_heads,
    d_ff        = d_ff,
    num_layers  = n_layers,
    dropout     = dropout,
    max_seq_len = block_size,   # RoPE tables precomputed up to this length
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}\n")

Model parameters: 804,289



## **Step 10.5: Architecture Smoke Test (RMSNorm + RoPE + SwiGLU + GQA)**

(run BEFORE the real training loop)

In [ ]:
"""
Checks:
  1. Only RMSNorm present (no LayerNorm)
  2. RoPE tables are fixed buffers, not learned parameters
  3. SwiGLU: W1/W2/W3 present, no ReLU
  4. GQA: W_K and W_V project to n_kv_heads * d_k (not full d_model)
  5. KV cache stores compact (n_kv_heads) K,V — not expanded num_heads
  6. Training-style forward pass shape correct
  7. Gradients flow through all new components
  8. Prefill + KV-cache decode (B=1) works correctly
"""

def run_smoke_test(model, vocab_size, d_model, n_heads, n_kv_heads):
    print("=" * 60)
    print("RUNNING ARCHITECTURE SMOKE TEST — v2")
    print("=" * 60)

    # ── Check 1: only RMSNorm ────────────────────────────────────
    norm_types = set(type(m).__name__ for m in model.modules() if "Norm" in type(m).__name__)
    print(f"[1] Norm classes: {norm_types}")
    assert norm_types == {"RMSNorm"}
    print("    PASS\n")

    # ── Check 2: RoPE buffers not parameters ────────────────────
    rope_in_params  = [n for n, _ in model.named_parameters()  if 'cos_table' in n or 'sin_table' in n]
    rope_in_buffers = [n for n, _ in model.named_buffers()     if 'cos_table' in n or 'sin_table' in n]
    print(f"[2] RoPE in params (want 0): {len(rope_in_params)}  |  in buffers (want >0): {len(rope_in_buffers)}")
    assert len(rope_in_params) == 0 and len(rope_in_buffers) > 0
    print("    PASS\n")

    # ── Check 3: SwiGLU, no ReLU ────────────────────────────────
    has_relu = any(isinstance(m, torch.nn.ReLU) for m in model.modules())
    swiglu_weights = [n for n, _ in model.named_parameters() if any(w in n for w in ['W1','W2','W3'])]
    print(f"[3] ReLU present (want False): {has_relu}  |  SwiGLU weights found: {len(swiglu_weights)}")
    assert not has_relu and len(swiglu_weights) > 0
    print("    PASS\n")

    # ── Check 4: GQA projection sizes ───────────────────────────
    d_k = d_model // n_heads
    first_block = model.blocks[0].attention
    wq_out = first_block.W_Q.out_features   # should be n_heads    * d_k = 128
    wk_out = first_block.W_K.out_features   # should be n_kv_heads * d_k = 64
    print(f"[4] W_Q out: {wq_out} (want {n_heads*d_k})  |  W_K out: {wk_out} (want {n_kv_heads*d_k})")
    assert wq_out == n_heads * d_k and wk_out == n_kv_heads * d_k
    print("    PASS — W_K/W_V project to smaller GQA dimension\n")

    # ── Check 5: KV cache shape is compact (n_kv_heads, not n_heads) ─
    model.eval()
    with torch.no_grad():
        dummy = torch.randint(0, vocab_size, (1, 5), device=device)
        _, kv_caches = model(dummy, kv_caches=None, start_pos=0)
        cached_k_heads = kv_caches[0]['K'].shape[1]
        print(f"[5] Cached K heads: {cached_k_heads} (want {n_kv_heads}, not {n_heads})")
        assert cached_k_heads == n_kv_heads
    print("    PASS — KV cache is compact (GQA groups, not full heads)\n")

    # ── Check 6: forward pass shape ─────────────────────────────
    model.train()
    dummy_ids = torch.randint(0, vocab_size, (4, 10), device=device)
    logits, _ = model(dummy_ids, kv_caches=None, start_pos=0)
    print(f"[6] Logits shape: {tuple(logits.shape)} (want (4, 10, {vocab_size}))")
    assert logits.shape == (4, 10, vocab_size)
    print("    PASS\n")

    # ── Check 7: gradients flow through all new components ───────
    targets = torch.randint(0, vocab_size, (4, 10), device=device)
    loss = torch.nn.CrossEntropyLoss()(logits.view(-1, vocab_size), targets.view(-1))
    loss.backward()
    keys_to_check = ['gamma', 'W1', 'W2', 'W3', 'W_Q', 'W_K', 'W_V']
    missing = [n for n, p in model.named_parameters()
               if p.grad is None and any(k in n for k in keys_to_check)]
    print(f"[7] Loss: {loss.item():.4f}  |  Missing grads: {missing}")
    assert not missing
    print("    PASS — gradients reached RMSNorm, SwiGLU, and GQA weights\n")
    model.zero_grad()

    # ── Check 8: prefill + decode path ──────────────────────────
    model.eval()
    with torch.no_grad():
        prompt = torch.randint(0, vocab_size, (1, 5), device=device)
        logits_pf, kv_caches = model(prompt, kv_caches=None, start_pos=0)
        assert logits_pf.shape == (1, 5, vocab_size)
        for step in range(3):
            tok = torch.randint(0, vocab_size, (1, 1), device=device)
            logits_dec, kv_caches = model(tok, kv_caches=kv_caches, start_pos=5+step)
            assert logits_dec.shape == (1, 1, vocab_size)
        print(f"[8] Prefill (T=5) + 3 decode steps — final shape {tuple(logits_dec.shape)}")
        print("    PASS — KV cache decode works with GQA\n")

    model.train()
    print("=" * 60)
    print("ALL SMOKE TESTS PASSED — full v2 architecture verified")
    print("=" * 60)


run_smoke_test(model, vocab_size, d_model, n_heads, n_kv_heads)

RUNNING ARCHITECTURE SMOKE TEST — v2
[1] Norm classes: {'RMSNorm'}
    PASS

[2] RoPE in params (want 0): 0  |  in buffers (want >0): 8
    PASS

[3] ReLU present (want False): False  |  SwiGLU weights found: 12
    PASS

[4] W_Q out: 128 (want 128)  |  W_K out: 64 (want 64)
    PASS — W_K/W_V project to smaller GQA dimension

[5] Cached K heads: 2 (want 2, not 4)
    PASS — KV cache is compact (GQA groups, not full heads)

[6] Logits shape: (4, 10, 65) (want (4, 10, 65))
    PASS

[7] Loss: 4.3565  |  Missing grads: []
    PASS — gradients reached RMSNorm, SwiGLU, and GQA weights

[8] Prefill (T=5) + 3 decode steps — final shape (1, 1, 65)
    PASS — KV cache decode works with GQA

ALL SMOKE TESTS PASSED — full v2 architecture verified


## **Step 11: The Training Loop**

### **Learning Rate Scheduling:**

- Phase 1 — Warmup (first ~5-10% of steps): Start the learning rate near zero and linearly ramp it up to the target value (your 3e-4). This protects the random, freshly-initialized weights from that dangerous first big step.
- Phase 2 — Cosine decay (remaining steps): Once warmup ends, decrease the learning rate smoothly following a cosine curve, from the peak value down to some small minimum, timed to reach that minimum right as training finishes. This lets the model move fast through the middle of training (when it's making the most progress) and slow down precisely as it's approaching a good solution, so it can settle in rather than overshoot.

In [ ]:
warmup_steps = int(0.1 * max_iters)   # 10% of training spent warming up
min_lr       = lr * 0.1               # decay down to 10% of peak, not all the way to 0

def get_lr(step):
    if step < warmup_steps:
        # Linear warmup: ramp from 0 up to peak lr
        return lr * (step / warmup_steps)
    # Cosine decay: peak lr down to min_lr over the remaining steps
    progress = (step - warmup_steps) / (max_iters - warmup_steps)
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * progress))

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn   = nn.CrossEntropyLoss()

@torch.no_grad()
def estimate_val_loss(eval_batches=20):
    """
    Run eval_batches batches through the model in eval mode
    and return the average validation loss.
    """
    model.eval()
    total_loss = 0.0
    for _ in range(eval_batches):
        xb, yb     = get_batch('val')
        logits, _  = model(xb)
        # logits: (B, T, vocab_size) → (B*T, vocab_size)
        # yb:     (B, T)             → (B*T,)
        loss       = loss_fn(logits.view(-1, vocab_size), yb.view(-1))
        total_loss += loss.item()
    model.train()
    return total_loss / eval_batches

print("Starting training...\n")

for step in range(max_iters):

    # ── Get a fresh random batch ────────────────────────────
    xb, yb = get_batch('train')           # xb: (B, T),  yb: (B, T)

    # ── LR Scheduling ───────────────────────────────────────
    current_lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr

    # ── Forward pass ────────────────────────────────────────
    logits, _ = model(xb)                 # logits: (B, T, vocab_size)

    # ── Compute loss ─────────────────────────────────────────
    # CrossEntropyLoss expects:
    #   input : (N, vocab_size)   — one row per prediction
    #   target: (N,)              — one int per prediction
    # So we flatten (B, T, vocab_size) → (B*T, vocab_size)
    #          and  (B, T)             → (B*T,)
    loss = loss_fn(
        logits.view(-1, vocab_size),      # (B*T, vocab_size)
        yb.view(-1)                       # (B*T,)
    )

    # ── Backward pass ────────────────────────────────────────
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # ── Logging ──────────────────────────────────────────────
    if step % eval_every == 0 or step == max_iters - 1:
        val_loss = estimate_val_loss()
        print(f"Step {step:4d} | Train loss: {loss.item():.4f} | Val loss: {val_loss:.4f}")

print("\nTraining complete.")

Starting training...

Step    0 | Train loss: 4.3277 | Val loss: 4.1157
Step  200 | Train loss: 2.4759 | Val loss: 2.4590
Step  400 | Train loss: 2.3509 | Val loss: 2.2776
Step  600 | Train loss: 2.2128 | Val loss: 2.1743
Step  800 | Train loss: 2.1447 | Val loss: 2.1280
Step 1000 | Train loss: 2.0394 | Val loss: 2.0672
Step 1200 | Train loss: 1.9734 | Val loss: 2.0056
Step 1400 | Train loss: 1.9212 | Val loss: 1.9575
Step 1600 | Train loss: 1.9239 | Val loss: 1.9344
Step 1800 | Train loss: 1.9024 | Val loss: 1.9054
Step 2000 | Train loss: 1.7873 | Val loss: 1.8902
Step 2200 | Train loss: 1.8567 | Val loss: 1.8527
Step 2400 | Train loss: 1.7833 | Val loss: 1.8526
Step 2600 | Train loss: 1.7789 | Val loss: 1.8348
Step 2800 | Train loss: 1.7192 | Val loss: 1.8229
Step 3000 | Train loss: 1.9392 | Val loss: 1.8124
Step 3200 | Train loss: 1.7603 | Val loss: 1.8127
Step 3400 | Train loss: 1.6949 | Val loss: 1.7911
Step 3600 | Train loss: 1.6893 | Val loss: 1.7729
Step 3800 | Train loss: 1.64

## **Step 12: Save the Model Weights**

In [ ]:
torch.save({
    'model_state_dict' : model.state_dict(),
    'vocab_size'       : vocab_size,
    'd_model'          : d_model,
    'n_heads'          : n_heads,
    'd_ff'             : d_ff,
    'n_layers'         : n_layers,
    'dropout'          : dropout,
    'stoi'             : stoi,
    'itos'             : itos,
}, 'minigpt_shakespeare.pt')

print("Model saved to minigpt_shakespeare.pt")

Model saved to minigpt_shakespeare.pt


## **Step 13: The Generation Flow**

In [ ]:
# ==============================================================================
# TOP-K APPROACH
# ==============================================================================
# def _sample(logits_1d, temperature, top_k, generated=None, rep_penalty=1.3):
#     # Penalise recently seen tokens
#     if generated and rep_penalty > 1.0:
#         for token_id in set(generated[-20:]):  # look at last 20 chars
#             logits_1d[token_id] /= rep_penalty  # divide logit → lower probability

#     logits_1d = logits_1d / temperature
#     if top_k is not None:
#         values, _ = torch.topk(logits_1d, top_k)
#         logits_1d[logits_1d < values[-1]] = float('-inf')

#     probs = torch.softmax(logits_1d, dim=-1)
#     return torch.multinomial(probs, num_samples=1).item()


# ==============================================================================
# TOP-P APPROACH
# ==============================================================================

def _sample(logits_1d, temperature, top_p=0.9, generated=None, rep_penalty=1.2):
    """
    Temperature + repetition penalty + nucleus (top-p) sampling.
    """
    # Step 1: Repetition penalty — suppress recently seen characters
    if generated and rep_penalty > 1.0:
        for token_id in set(generated[-15:]):
            logits_1d[token_id] /= rep_penalty

    # Temperature = 0 → pure greedy
    if temperature == 0:
        return logits_1d.argmax().item()

    # Step 2: Temperature scaling
    logits_1d = logits_1d / temperature

    # Step 3: Convert to probabilities
    probs = torch.softmax(logits_1d, dim=-1)

    # Step 4: Top-p nucleus sampling
    # Sort probabilities descending
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)

    # Compute cumulative probabilities
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Find cutoff: remove tokens once cumulative prob exceeds p
    # shift by 1 so we always keep at least the top token
    sorted_indices_to_remove = cumulative_probs - sorted_probs > top_p

    # Zero out the removed tokens
    sorted_probs[sorted_indices_to_remove] = 0.0

    # Renormalise so remaining probs sum to 1
    sorted_probs = sorted_probs / sorted_probs.sum()

    # Sample from the nucleus
    sampled_idx = torch.multinomial(sorted_probs, num_samples=1)

    # Map back to original token index
    return sorted_indices[sampled_idx].item()

In [ ]:
def generate(model, prompt, max_new_tokens=200, temperature=0.6, top_p=0.9, rep_penalty=1.2):
    model.eval()
    token_ids = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
    generated = token_ids[0].tolist()

    with torch.no_grad():

        # Prefill
        logits, kv_caches = model(token_ids, kv_caches=None, start_pos=0)
        next_id = _sample(logits[0, -1], temperature, top_p=top_p,
                          generated=generated, rep_penalty=rep_penalty)
        generated.append(next_id)

        # Decode
        for _ in range(max_new_tokens - 1):
            current_pos  = len(generated) - 1
            input_tensor = torch.tensor([[next_id]], dtype=torch.long).to(device)
            logits, kv_caches = model(input_tensor, kv_caches=kv_caches,
                                      start_pos=current_pos)
            next_id = _sample(logits[0, -1], temperature, top_p=top_p,
                              generated=generated, rep_penalty=rep_penalty)
            generated.append(next_id)

    return decode(generated)

In [ ]:
print("\n=== Sample generation ===\n")
output = generate(
    model,
    prompt         = "To be or",
    max_new_tokens = 200,
    temperature    = 0.6,
    top_p          = 0.9,
    rep_penalty    = 1.2
)
print(output)


=== Sample generation ===

To be ord,
And like as of the fair words to me,
His thy was and suchomeeerrphallllee t on mid alere me, in alerrt alallllame iontow h uck w hellallllllly minge ouck th te at te aly aten on he, y w alis ton al
